# Getting Started with fenics-pipeline

Welcome. This notebook is the front door to the project. If you can run every
cell below top-to-bottom without error, your environment is set up correctly
and you've just produced your first topology-optimized 3D-printable part.

**What this notebook does:**

1. Verifies your environment (Docker, the Rust solver binary, parameter files).
2. Gives you a tour of the pipeline stages with a small visual map.
3. Lets you pick one of the included parts and run the full pipeline end-to-end.
4. Surfaces the outputs — convergence plots, stress heatmaps, the final STL —
   inline so you don't have to go digging in `outputs/`.

**Before you start, make sure:**

- You're running this notebook inside the `fenics-pipeline` JupyterLab
  container (URL should be `localhost:8888`).
- The kernel at the top-right reads **FEniCSx Pipeline**. If it says "Python 3"
  or anything else, click it and switch.
- You've built the Rust solver binary and it sits at `bin/simp_solver`. If
  you haven't, see **Part 6** of `README_HOBBYIST.md` — it's a one-time
  `cargo build --release` and a file copy.

If you've never touched this repo before, work through the hobbyist README
first, then come back here.

---


## 1. Environment check

This cell confirms everything is where it needs to be. If anything prints a ✗, fix it before moving on — the rest of the notebook will fail if you skip this.


In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

# Running inside the container, everything is rooted at /workspace
os.chdir("/workspace")
sys.path.insert(0, "/workspace")

PASS = "\u2713"   # check
FAIL = "\u2717"   # x
WARN = "\u26a0"   # warning

errors = 0

def check(label, ok, hint=""):
    global errors
    mark = PASS if ok else FAIL
    print(f"  {mark} {label}")
    if not ok:
        errors += 1
        if hint:
            print(f"      \u2192 {hint}")

print("\u2500\u2500 environment \u2500\u2500")

# Python kernel
check(f"Python: {sys.version.split()[0]}", sys.version_info >= (3, 10))

# Core libs
try:
    import dolfinx
    check(f"dolfinx: {dolfinx.__version__}", True)
except ImportError:
    check("dolfinx", False, "Wrong kernel. Switch to 'FEniCSx Pipeline' at top-right.")

try:
    import gmsh
    check(f"gmsh: {gmsh.__version__ if hasattr(gmsh, '__version__') else 'installed'}", True)
except ImportError:
    check("gmsh", False)

try:
    import trimesh, skimage, papermill
    check(f"trimesh: {trimesh.__version__}", True)
    check(f"scikit-image: {skimage.__version__}", True)
    check(f"papermill: {papermill.__version__}", True)
except ImportError as e:
    check(f"scientific stack ({e})", False)

# OpenSCAD
check("OpenSCAD on PATH", shutil.which("openscad") is not None,
      "Container was built without OpenSCAD \u2014 rebuild with `make build`.")

# Rust solver binary
solver = Path("bin/simp_solver")
check(f"Rust solver at {solver}", solver.exists() and os.access(solver, os.X_OK),
      "Build it: on your WSL2 host, run `cd solver && cargo build --release && cp target/release/simp_solver ../bin/` and restart this cell.")

# Parameter files
scad = Path("scad")
params = list(scad.glob("*_params.json"))
check(f"{len(params)} parametric parts in scad/", len(params) >= 1)
for p in params:
    print(f"      \u2022 {p.stem.replace('_params', '')}")

# Output tree
for sub in ["outputs/meshes", "outputs/stl", "outputs/reports", "outputs/executed_nbs", "outputs/problem"]:
    Path(sub).mkdir(parents=True, exist_ok=True)
check("outputs/ tree ready", True)

print()
if errors:
    print(f"{FAIL} {errors} issue(s) above \u2014 fix before continuing.")
else:
    print(f"{PASS} all checks passed \u2014 you're ready to run the pipeline.")


---

## 2. How the pipeline fits together

Five stages, each a notebook under `notebooks/`. Each stage reads a handoff
JSON from the previous stage and writes its own — so stages are loosely
coupled and you can re-run any one of them without rerunning the others.

```
    scad/<part>.scad           ─▶ [01_geometry_openscad]   ─▶  .stl
                                                                ↓
                                       [02_mesh_gmsh]           ─▶  .msh + .xdmf
                                                                ↓
                                       [03_fea_fenicsx]         ─▶  displacement + stress
                                                                ↓
    bin/simp_solver            ─▶ [04_simp_optimization]   ─▶  density field
                                                                ↓
                                       [05_stl_export]          ─▶  outputs/meshes/<part>_optimized.stl
```

**Key files and where they live:**

| | |
|---|---|
| Parametric geometry | `scad/<part>.scad` + `scad/<part>_params.json` |
| Stage notebooks | `notebooks/0[1-5]_*.ipynb` |
| Headless orchestrator | `notebooks/pipeline_full.ipynb` |
| Rust SIMP solver binary | `bin/simp_solver` |
| Rust solver source | `solver/src/*.rs` |
| Python pipeline modules | `src/{geometry,meshing,fea,optimization}/` |
| Everything produced by a run | `outputs/` |

The optimizer itself (stage 4) is the most involved stage. It's a standalone
Rust binary — `bin/simp_solver` — invoked from the notebook with a JSON
problem description. The notebook hands it voxelized geometry, a load case,
and SIMP hyperparameters; the binary writes back a density field and a
run log. This separation is why the optimization loop can run at compiled
speeds even though the surrounding pipeline is Python.

---


## 3. Pick a part and load its parameters

Four example parts ship with the repo. Start with `base_part` on your first run — it's the fastest to converge and the easiest to eyeball for sanity. Switch to a different one by editing the `PART_NAME` variable below.


In [ ]:
import json
from pathlib import Path

# \u2500\u2500\u2500 CHANGE ME TO TRY A DIFFERENT PART \u2500\u2500\u2500
PART_NAME = "base_part"
# Options: "base_part", "cantilever_arm", "motor_mount", "tripod_mount_base"

params_file = Path(f"scad/{PART_NAME}_params.json")
assert params_file.exists(), f"No params file for {PART_NAME!r}. Check scad/ for available parts."

with open(params_file) as f:
    params = json.load(f)

# Copy to scad/params.json \u2014 this is what the pipeline reads
Path("scad/params.json").write_text(json.dumps(params, indent=2))
print(f"Loaded: {params_file}")
print(f"Activated as: scad/params.json")
print()

# Show the interesting bits
geom = params.get("geometry", {})
load = params.get("load_case", {}).get("load", {})
fixed = params.get("load_case", {}).get("fixed", {})
mesh = params.get("mesh_hints", {})

print(f"Part name:          {params['part_name']}")
print(f"Bounding box (mm):  {geom.get('length', '?')} \u00d7 {geom.get('width', '?')} \u00d7 {geom.get('height', '?')}")
print(f"Mesh element size:  {mesh.get('target_element_size', '?')} mm")
print(f"Load:               {load.get('magnitude_n', '?')} N, direction {load.get('direction', '?')}, on face {load.get('face', '?')}")
print(f"Fixed:              face {fixed.get('face', '?')}, selector {fixed.get('selector', 'full-face')}")


### What those knobs mean

- **Mesh element size** — smaller values mean a finer mesh, which means
  more accurate FEA and smoother optimized geometry, but also *dramatically*
  more memory and compute. Default values are tuned to finish in ~10 min on
  an 8-core CPU. Halving the element size usually 8×s the element count.
- **Load magnitude** — the force applied to the loaded face, in newtons.
  10 kN is a reasonable "this part is under real stress" value for a bracket.
- **Load direction** — a unit vector. `[0, 0, -1]` pushes down along −Z.
- **Fixed face / selector** — which face is bolted down, and whether
  the fix is the whole face, just the corners, or something more specific.
  `base_part` has `corners` (4 mounting holes pinned), which is the most
  realistic condition for a bolted bracket.

---


## 4. Run the full pipeline

The cell below calls Papermill to execute all five stage notebooks in order. This is the same thing `make run` does from a shell — we're just doing it from here so you can see the output inline.

**Expect 5–15 minutes depending on your CPU.** Progress prints as each stage finishes; you can keep an eye on `outputs/executed_nbs/` if you want to watch the intermediate notebooks as they're written.

If you'd rather run each stage one notebook at a time the first time through (recommended if you're learning the pipeline), skip this cell and open the numbered notebooks in `notebooks/` in order instead.


In [ ]:
import papermill as pm
from datetime import datetime
from pathlib import Path

executed_dir = Path("outputs/executed_nbs")
executed_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_nb = executed_dir / f"pipeline_full_{PART_NAME}_{ts}.ipynb"

print(f"Running pipeline for: {PART_NAME}")
print(f"Output notebook:      {out_nb}")
print(f"Started:              {datetime.now().strftime('%H:%M:%S')}")
print()

try:
    pm.execute_notebook(
        input_path="notebooks/pipeline_full.ipynb",
        output_path=str(out_nb),
        parameters={
            "PART_NAME": PART_NAME,
            "PARAMS_JSON": "scad/params.json",
            "OUTPUT_DIR": "outputs",
            "KERNEL_NAME": "fenics-pipeline",
        },
        kernel_name="fenics-pipeline",
        progress_bar=True,
        log_output=True,
    )
    print()
    print(f"\u2713 pipeline finished at {datetime.now().strftime('%H:%M:%S')}")
except pm.PapermillExecutionError as e:
    print()
    print(f"\u2717 failed at stage: {e.cell.metadata.get('papermill', {}).get('cell_id', 'unknown')}")
    print(f"   {e.ename}: {e.evalue}")
    print()
    print(f"Full traceback in: {out_nb}")
    raise


---

## 5. Look at what it produced

Assuming the run succeeded, there are a few things worth eyeballing before
you go straight to the STL. The convergence curve tells you whether the
optimizer actually converged or bailed at the iteration cap; the before/after
PNG is a quick sanity check on the shape; and the stress heatmap tells you
whether your load case made physical sense.



In [ ]:
from pathlib import Path
from IPython.display import Image, display, Markdown

reports = Path("outputs/reports")
meshes = Path("outputs/meshes")
stl_out = Path("outputs/stl")

# The report files we want to surface, in an opinionated order
report_files = [
    (f"{PART_NAME}_geometry.png",         "Input geometry (what stage 1 produced)"),
    (f"{PART_NAME}_mesh.png",             "Meshed part (what stage 2 produced)"),
    (f"{PART_NAME}_stress_heatmap.png",   "Von Mises stress under load (stage 3) \u2014 red = highly stressed"),
    (f"{PART_NAME}_convergence.png",      "Optimizer convergence (stage 4) \u2014 should flatten out"),
    (f"{PART_NAME}_density_histogram.png","Final density distribution \u2014 should be bimodal at 0 and 1"),
    (f"{PART_NAME}_before_after.png",     "Before vs. after"),
    (f"{PART_NAME}_stl_wireframe.png",    "Final STL wireframe"),
]

for fname, caption in report_files:
    path = reports / fname
    if path.exists():
        display(Markdown(f"**{caption}**  \n`{path}`"))
        display(Image(filename=str(path)))
    else:
        display(Markdown(f"\u26a0 `{path}` not produced by this run"))


## 6. Find your STL

The final exported STL lives in `outputs/meshes/` (or `outputs/stl/` for some part configurations). Here's what the run produced:


In [ ]:
from pathlib import Path

candidates = list(Path("outputs/meshes").glob(f"{PART_NAME}*.stl")) + \
             list(Path("outputs/stl").glob(f"{PART_NAME}*.stl"))

if not candidates:
    print(f"No STLs found for {PART_NAME}. Something in stage 5 didn't write a file \u2014 check the executed notebook.")
else:
    print(f"STL files for {PART_NAME}:")
    for stl in sorted(candidates):
        size_kb = stl.stat().st_size / 1024
        print(f"  \u2022 {stl}   ({size_kb:,.1f} KB)")
    print()
    primary = next((s for s in candidates if "optimized" in s.name), candidates[0])
    print(f"Drop into your slicer: {primary}")
    print()
    print("To copy out of WSL2 onto Windows:")
    print(f"  cp {primary} /mnt/c/Users/YOUR_WINDOWS_USER/Desktop/")


## 7. Optional: preview the STL inline

A quick inline look with trimesh to sanity-check the mesh before you slice it.
This renders a static image of the STL from a default viewpoint and reports
whether it's watertight (the one thing that matters for printing).


In [ ]:
import trimesh
from pathlib import Path
from IPython.display import display

stl_candidates = list(Path("outputs/meshes").glob(f"{PART_NAME}*optimized*.stl")) + \
                 list(Path("outputs/meshes").glob(f"{PART_NAME}*.stl"))

if stl_candidates:
    mesh = trimesh.load(str(stl_candidates[0]))
    print(f"File:             {stl_candidates[0]}")
    print(f"Vertices:         {len(mesh.vertices):,}")
    print(f"Faces:            {len(mesh.faces):,}")
    print(f"Watertight:       {mesh.is_watertight}")
    print(f"Bounding box mm:  {mesh.bounds[1] - mesh.bounds[0]}")
    print(f"Volume cm\u00b3:        {mesh.volume / 1000:.2f}")
    print()
    if not mesh.is_watertight:
        print("\u26a0 Not watertight \u2014 most slicers will still print it but repair it first")
        print("  to be safe. PrusaSlicer, Cura, and Bambu Studio all have built-in repair.")
else:
    print(f"No STL to preview for {PART_NAME}")


---

## Where to go from here

**Run a different part.** Change `PART_NAME` in cell 3 to `motor_mount`,
`cantilever_arm`, or `tripod_mount_base`. Re-run cells 3–6.

**Change the load.** Open the active `scad/params.json` in JupyterLab
(left sidebar), edit the `load_case` block, save, and re-run from cell 4.
You don't need to re-import the params.

**Run stages individually.** Open the notebooks in `notebooks/` one at a
time. Each one is self-contained and re-reads the previous stage's handoff
from `outputs/meshes/<part>_stage0*.json`. This is the right way to work
when you're iterating on a single stage.

**Tune the optimizer.** `notebooks/04_simp_optimization.ipynb` Cell 0 has
`VOLUME_FRACTION`, `FILTER_RADIUS`, `PENAL`, and `MAX_ITERATIONS`. Lower
volume fraction = lighter part; higher filter radius = smoother but less
material savings. See the hobbyist README for the cookbook.

**Bring your own STL.** If you already designed a part in Fusion 360 /
FreeCAD / SolidWorks / etc., export it and skip stage 1:

```bash
make import-stl STL=/path/to/my_part.stl PART=my_part SIZE=2.0 FACE=top LOAD=5000
```

Then open `notebooks/02_mesh_gmsh.ipynb` and run all cells.

**Read the code.** Start with `solver/src/simp.rs` for the optimization loop,
`src/geometry/region_factory.py` for how the declarative params become
voxelized constraints, and `notebooks/04_simp_optimization.ipynb` for how
the two are glued together.

Good luck. If something breaks, the issues tracker is the right place to
report it.
